# **Import Libraries**

In [ ]:
import time
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# **Load Data**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Project Datasets/Global Air Pollution Dataset.csv')
print(f"Dataset shape: {df.shape}")
df.head(10)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset shape: (23463, 12)


,Country,City,AQI Value,AQI Category,CO AQI Value,CO AQI Category,Ozone AQI Value,Ozone AQI Category,NO2 AQI Value,NO2 AQI Category,PM2.5 AQI Value,PM2.5 AQI Category
0,Russian Federation,Praskoveya,51,Moderate,1,Good,36,Good,0,Good,51,Moderate
1,Brazil,Presidente Dutra,41,Good,1,Good,5,Good,1,Good,41,Good
2,Italy,Priolo Gargallo,66,Moderate,1,Good,39,Good,2,Good,66,Moderate
3,Poland,Przasnysz,34,Good,1,Good,34,Good,0,Good,20,Good
4,France,Punaauia,22,Good,0,Good,22,Good,0,Good,6,Good
5,United States of America,Punta Gorda,54,Moderate,1,Good,14,Good,11,Good,54,Moderate
6,Germany,Puttlingen,62,Moderate,1,Good,35,Good,3,Good,62,Moderate
7,Belgium,Puurs,64,Moderate,1,Good,29,Good,7,Good,64,Moderate
8,Russian Federation,Pyatigorsk,54,Moderate,1,Good,41,Good,1,Good,54,Moderate
9,Egypt,Qalyub,142,Unhealthy for Sensitive Groups,3,Good,89,Moderate,9,Good,142,Unhealthy for Sensitive Groups


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23463 entries, 0 to 23462
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Country             23036 non-null  object
 1   City                23462 non-null  object
 2   AQI Value           23463 non-null  int64 
 3   AQI Category        23463 non-null  object
 4   CO AQI Value        23463 non-null  int64 
 5   CO AQI Category     23463 non-null  object
 6   Ozone AQI Value     23463 non-null  int64 
 7   Ozone AQI Category  23463 non-null  object
 8   NO2 AQI Value       23463 non-null  int64 
 9   NO2 AQI Category    23463 non-null  object
 10  PM2.5 AQI Value     23463 non-null  int64 
 11  PM2.5 AQI Category  23463 non-null  object
dtypes: int64(5), object(7)
memory usage: 2.1+ MB


# **Check for Duplicate Rows**

In [ ]:
print(f"Duplicate rows: {df.duplicated().sum()}")

Duplicate rows: 0


# **Handle Missing Values**

In [ ]:
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0]
print("Missing values in each column:")
print(missing_values)

Missing values in each column:
Country    427
City         1
dtype: int64


In [ ]:
df = df.dropna(subset=["City"]).reset_index(drop=True)
print(f"\nAfter dropping missing-City rows: {df.shape}")


After dropping missing-City rows: (23462, 12)


# **Fill in Missing 'Country' Values**

In [ ]:
geolocator = Nominatim(user_agent="geoapi_exercise")

def get_country(city):
    try:
        location = geolocator.geocode(city, language='en', timeout=5)
        if location and location.address:
            address_parts = location.address.split(', ')
            # Attempt to extract country from the address parts
            # This can be tricky, often the last part is the country
            country = address_parts[-1] # Default to last part
            # Refine for common cases or more robust parsing if needed
            # Example: if 'United States' is in address, use that.
            # For now, let's try to get a more reliable country from raw data
            raw_data_country = location.raw.get('address', {}).get('country')
            if raw_data_country:
                return raw_data_country
            return country
    except GeocoderTimedOut:
        print(f"Geocoder timed out for city: {city}. Retrying...")
        time.sleep(5) # Wait for 5 seconds before retrying
        return get_country(city) # Retry recursively
    except GeocoderServiceError as e:
        print(f"Geocoder service error for city: {city}. Error: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred for city: {city}. Error: {e}")
        return None
    return None


# Identify rows where 'Country' is missing but 'City' is present
missing_country_rows = df[df['Country'].isnull() & df['City'].notnull()].index

print(f"Found {len(missing_country_rows)} rows with missing Country values to process.")

# Iterate and fill missing countries
for i, index in enumerate(missing_country_rows):
    city = df.loc[index, 'City']
    if city:
        print(f"[{i+1}/{len(missing_country_rows)}] Getting country for city: {city}")
        country = get_country(city)
        if country:
            df.loc[index, 'Country'] = country
            print(f"  -> Found country: {country}")
        else:
            print(f"  -> Could not determine country for city: {city}")
    time.sleep(1) # Add a delay to respect API rate limits

print(f"\nMissing 'Country' values after filling: {df['Country'].isnull().sum()}")
print(f"Missing 'City' values after filling: {df['City'].isnull().sum()}")

Found 427 rows with missing Country values to process.
[1/427] Getting country for city: Granville
  -> Found country: France
[2/427] Getting country for city: Kingston Upon Hull
  -> Found country: United Kingdom
[3/427] Getting country for city: New Waterford
  -> Found country: United States
[4/427] Getting country for city: Kingstown
  -> Found country: Saint Vincent and the Grenadines
[5/427] Getting country for city: Nanakuli
  -> Found country: United States
[6/427] Getting country for city: Lavagna
  -> Found country: Italy
[7/427] Getting country for city: Ladispoli
  -> Found country: Italy
[8/427] Getting country for city: Dong Hoi
  -> Found country: Vietnam
[9/427] Getting country for city: Nettuno
  -> Found country: Italy
[10/427] Getting country for city: Puebloviejo
  -> Found country: Ecuador
[11/427] Getting country for city: Fiumicino
  -> Found country: Italy
[12/427] Getting country for city: Nishinomiya
  -> Found country: Japan
[13/427] Getting country for city:

In [ ]:
df = df.dropna(subset=['Country']).reset_index(drop=True)
print(f"\nAfter dropping rows with missing Country values: {df.shape}")


After dropping rows with missing Country values: (23457, 12)


# **Save Cleaned Dataset**

In [ ]:
output_path = '/content/drive/MyDrive/Project Datasets/Global Air Pollution Dataset_cleaned.csv'
df.to_csv(output_path, index=False)
print(f"Cleaned DataFrame saved to: {output_path}")

Cleaned DataFrame saved to: /content/drive/MyDrive/Project Datasets/Global Air Pollution Dataset_cleaned.csv
